# DCGAN (Deep Convolutional GAN)

This is a DCGAN implementation with Keras 3 (PyTorch backend), trained with Fashion-MNIST to generate gray-scale 28x28 images. Link to the DCGAN Paper: [Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks](https://arxiv.org/abs/1511.06434)

Training takes a while, so run this if you have a GPU! On a Tesla T4 GPU, one epoch takes about 80 seconds, i.e. 50 epochs would take about 70 minutes. However, already after a few epochs, the generated images look like clothes.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fpschill/cas_main_dl_2026/blob/main/08_dcgan.ipynb)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # must be set BEFORE keras is imported
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import torch

import keras
from keras import layers
from keras import ops
from keras.models import Sequential

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())
print("GPU available:", torch.cuda.is_available())

# folders for the generated images and the generator weights
IMAGE_DIR = "dcgan_images"
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

### Load Data

In [ ]:
(train_images, train_labels), (_, _) = keras.datasets.fashion_mnist.load_data()
train_images.shape

### Display Data

In [ ]:
# We plot the first 25 images of the training dataset
plt.figure(figsize =(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap = "gray")
plt.show()

### Preprocessing

In [ ]:
# Reshape data and also convert the data type to 'float32'
print(train_images.shape)
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1).astype('float32')
print(train_images.shape)

# Normalize the images to [-1, 1] which is the range of the tanh activation
train_images = (train_images - 127.5) / 127.5

### Generator Model

In [ ]:
# latent dimension of the random noise
LATENT_DIM = 100
# weight initializer for G per DCGAN paper
WEIGHT_INIT = keras.initializers.RandomNormal(mean=0.0, stddev=0.02)
# number of channels, 1 for gray scale and 3 for color images
CHANNELS = 1

In [ ]:
def build_generator():
    # create a Keras Sequential model
    model = Sequential(name="generator")

    # input: random noise vector of size LATENT_DIM
    model.add(layers.Input(shape=(LATENT_DIM,)))

    # prepare for reshape: FC => BN => RN layers
    model.add(layers.Dense(7 * 7 * 256))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # 1D => 3D: reshape the output of the previous layer
    model.add(layers.Reshape((7, 7, 256)))

    # upsample to 14x14: apply a transposed CONV => BN => RELU
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2),padding="same", kernel_initializer=WEIGHT_INIT))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # upsample to 28x28: apply a transposed CONV => BN => RELU
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2),padding="same", kernel_initializer=WEIGHT_INIT))
    model.add(layers.BatchNormalization())
    model.add(layers.ReLU())

    # final layer: Conv2D with tanh activation
    model.add(layers.Conv2D(CHANNELS, (5, 5), padding="same", activation="tanh"))

    # return the generator model
    return model

# build the generator model
generator = build_generator()

generator.summary()

### Discriminator Model

In [ ]:
def build_discriminator(width, height, depth, alpha=0.2):
    # create a Keras Sequential model
    model = Sequential(name="discriminator")
    input_shape = (height, width, depth)

    model.add(layers.Input(shape=input_shape))

    # first set of CONV => BN => leaky ReLU layers
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(negative_slope=alpha))

    # second set of CONV => BN => leaky ReLU layers
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding="same"))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU(negative_slope=alpha))

    # flatten and apply dropout
    model.add(layers.Flatten())
    model.add(layers.Dropout(0.3))

    # sigmoid in the last layer outputs a single value for binary classification
    model.add(layers.Dense(1, activation="sigmoid"))

    # return the discriminator model
    return model

# build the discriminator model
discriminator = build_discriminator(28, 28, 1)

discriminator.summary()

### DCGAN class: override train_step

In each training step, first the discriminator and then the generator is trained. With the PyTorch backend, we compute the gradients ourselves in `train_step`:
- `self.zero_grad()` sets the gradients of all weights to zero
- `loss.backward()` computes the gradients of the loss with respect to all weights
- `optimizer.apply(grads, weights)` updates the weights of the discriminator or of the generator only

In [ ]:
class DCGAN(keras.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        self.seed_generator = keras.random.SeedGenerator(1337)
        self.d_loss_metric = keras.metrics.Mean(name="d_loss")
        self.g_loss_metric = keras.metrics.Mean(name="g_loss")
        # the generator and the discriminator are already built, so the DCGAN model is built too
        self.built = True

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super().compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    def train_step(self, real_images):
        real_images = ops.convert_to_tensor(real_images)
        batch_size = ops.shape(real_images)[0]
        noise = keras.random.normal(shape=(batch_size, self.latent_dim), seed=self.seed_generator)

        # Step 1. Train the discriminator with both real images (label as 1) and fake images (classified as label as 0)
        self.zero_grad()
        # Compute discriminator loss on real images
        pred_real = self.discriminator(real_images, training=True)
        d_loss_real = self.loss_fn(ops.ones((batch_size, 1)), pred_real)

        # Compute discriminator loss on fake images
        # (stop_gradient: in this step, no gradients are needed for the generator)
        fake_images = ops.stop_gradient(self.generator(noise, training=True))
        pred_fake = self.discriminator(fake_images, training=True)
        d_loss_fake = self.loss_fn(ops.zeros((batch_size, 1)), pred_fake)

        # total discriminator loss
        d_loss = (d_loss_real + d_loss_fake)/2
        # Compute discriminator gradients
        d_loss.backward()
        grads = [v.value.grad for v in self.discriminator.trainable_weights]
        # Update discriminator weights
        with torch.no_grad():
            self.d_optimizer.apply(grads, self.discriminator.trainable_weights)

        # Step 2. Train the generator (do not update weights of the discriminator)
        # G wants D to think the fake images are real (label as 1)
        misleading_labels = ops.ones((batch_size, 1))

        self.zero_grad()
        fake_images = self.generator(noise, training=True)
        pred_fake = self.discriminator(fake_images, training=True)
        g_loss = self.loss_fn(misleading_labels, pred_fake)
        # Compute generator gradients
        g_loss.backward()
        grads = [v.value.grad for v in self.generator.trainable_weights]
        # Update generator weights
        with torch.no_grad():
            self.g_optimizer.apply(grads, self.generator.trainable_weights)

        self.d_loss_metric.update_state(d_loss)
        self.g_loss_metric.update_state(g_loss)

        return {"d_loss": self.d_loss_metric.result(), "g_loss": self.g_loss_metric.result()}

### Use Keras Callback to monitor training

In [ ]:
class GANMonitor(keras.callbacks.Callback):
    def __init__(self, num_img=3, latent_dim=100):
        super().__init__()
        self.num_img = num_img
        self.latent_dim = latent_dim
        # Create random noise seed for visualization during training
        self.seed = keras.random.normal((self.num_img, latent_dim))

    def on_epoch_end(self, epoch, logs=None):
        generated_images = self.model.generator(self.seed, training=False)
        generated_images = ops.convert_to_numpy(generated_images * 127.5 + 127.5)

        fig = plt.figure(figsize=(4, 4))
        for i in range(self.num_img):
            plt.subplot(4, 4, i+1)
            img = keras.utils.array_to_img(generated_images[i])
            plt.imshow(img, cmap='gray')
            plt.axis('off')
        plt.savefig(os.path.join(IMAGE_DIR, 'epoch_{:03d}.png'.format(epoch)))
        plt.show()

    def on_train_end(self, logs=None):
        self.model.generator.save_weights('checkpoints/dcgan_generator.weights.h5')

### Create DCGAN and compile model

In [ ]:
dcgan = DCGAN(discriminator=discriminator, generator=generator, latent_dim=LATENT_DIM)

In [ ]:
LR = 0.0002 # learning rate

dcgan.compile(
    d_optimizer=keras.optimizers.Adam(learning_rate=LR, beta_1=0.5),
    g_optimizer=keras.optimizers.Adam(learning_rate=LR, beta_1=0.5),
    loss_fn=keras.losses.BinaryCrossentropy(),
)

### Train model

In [ ]:
NUM_EPOCHS = 5 # number of epochs
dcgan.fit(train_images, epochs=NUM_EPOCHS, callbacks=[GANMonitor(num_img=16, latent_dim=LATENT_DIM)])

### Create animation

In [ ]:
import imageio
images = []
for i in range(NUM_EPOCHS):
    file = os.path.join(IMAGE_DIR, "epoch_{:03d}.png".format(i))
    if os.path.isfile(file):
        images.append(imageio.v2.imread(file))
imageio.mimsave(os.path.join(IMAGE_DIR, 'movie.gif'), images, 'GIF', duration=1000, loop=1)

In [ ]:
from IPython.display import Image
Image(filename=os.path.join(IMAGE_DIR, "movie.gif"))

### Further Examples

Another DCGAN example trained on face images (CELEB-A dataset) can be found at:
https://keras.io/examples/generative/dcgan_overriding_train_step/

Many more KERAS examples for generative models can be found here:
https://keras.io/examples/generative/